# OVITO Basic force-chain export using bond types

This version is designed for **OVITO Basic / free**. Since LAMMPS `.data` files preserve bond topology and **Bond Type** but do not preserve arbitrary bond properties such as Force, Betweenness, or ForceBetweennessScore, this notebook bins the chosen metric into bond types.

It exports three `.data` files:

- `force_chains_by_force.data`
- `force_chains_by_betweenness.data`
- `force_chains_by_score.data`

In OVITO, open one of these files, then use:

`Add modification → Color coding → Operate on: Bonds → Input property: Bond Type`


In [ ]:
from ovito.io import import_file, export_file
import numpy as np
import pandas as pd
import networkx as nx

# =========================
# FILES
# =========================
# Put this notebook in the same folder as these files, or replace with full paths.
PARTICLE_FILE = "dump_mu0.7_mur0.1_mut0.1.cor"
CONTACT_FILE  = "dump_mu0.7_mur0.1_mut0.1.lst"

# =========================
# SETTINGS
# =========================
CHAIN_PERCENTILE = 90   # top 10% edge-betweenness contacts become force-chain bonds
N_BOND_TYPES     = 10    # number of color/type bins for OVITO Basic
COLOR_BY         = "score"  # changed automatically by export loop below


def read_lammps_local_contacts(filename):
    """
    Reads your dump.lst format:

    ITEM: ENTRIES index c_c_pairs[1] c_c_pairs[2]
                  c_c_forces[1] ... c_c_forces[8]

    where:
      c_c_pairs[1], c_c_pairs[2] = particle IDs
      c_c_forces[1:3] = force components fx, fy, fz
      c_c_forces[7]   = force magnitude |F|
      c_c_forces[8]   = torque magnitude |T|
    """
    rows = []
    reading = False

    with open(filename, "r") as f:
        for line in f:
            line = line.strip()

            if line.startswith("ITEM: ENTRIES"):
                reading = True
                continue

            if line.startswith("ITEM:") and reading:
                break

            if reading and line:
                parts = line.split()
                if len(parts) >= 11:
                    rows.append(parts)

    if not rows:
        raise RuntimeError(f"No contact rows found in {filename}")

    df = pd.DataFrame(rows).astype(float)

    contacts = pd.DataFrame()
    contacts["i"] = df[1].astype(int)
    contacts["j"] = df[2].astype(int)

    contacts["fx"] = df[3]
    contacts["fy"] = df[4]
    contacts["fz"] = df[5]

    # c_c_forces[7] = force magnitude in your dump
    contacts["F"] = df[9]

    return contacts


def compute_betweenness(contacts):
    """
    Computes edge betweenness using distance = 1/F.
    Large force means short network distance.
    """
    G = nx.Graph()

    for row in contacts.itertuples(index=False):
        i = int(row.i)
        j = int(row.j)
        F = float(row.F)

        if F > 1e-12:
            G.add_edge(i, j, weight=1.0 / F)

    B = nx.edge_betweenness_centrality(G, weight="weight")

    # Convert to dataframe and duplicate reversed edges for easy merge lookup.
    edges = []
    for (i, j), val in B.items():
        edges.append((i, j, val))
        edges.append((j, i, val))

    return pd.DataFrame(edges, columns=["i", "j", "B"])


def make_bond_types(values, n_bins=6):
    """
    Convert a continuous metric into integer bond types 1..n_bins.

    OVITO Basic can color bonds by Bond Type
    Type 1 = lowest metric bin; Type n_bins = highest metric bin.
    """
    values = np.asarray(values, dtype=float)

    if len(values) == 0:
        return np.array([], dtype=np.int32), np.array([])

    # Percentile bins give roughly even numbers of bonds per color/type.
    bin_edges = np.percentile(values, np.linspace(0, 100, n_bins + 1))

    # Avoid issues if repeated values make duplicate bin edges.
    bin_edges = np.asarray(bin_edges, dtype=float)
    for k in range(1, len(bin_edges)):
        if bin_edges[k] <= bin_edges[k - 1]:
            bin_edges[k] = bin_edges[k - 1] + 1e-12

    bond_types = np.digitize(values, bin_edges[1:-1], right=False) + 1
    bond_types = np.clip(bond_types, 1, n_bins)

    return bond_types.astype(np.int32), bin_edges


def choose_metric(color_by, bond_force, bond_betweenness, bond_score):
    """Select which quantity gets encoded into Bond Type."""
    if color_by == "force":
        return np.asarray(bond_force, dtype=float), "Force"
    elif color_by == "betweenness":
        return np.asarray(bond_betweenness, dtype=float), "Betweenness"
    elif color_by == "score":
        return np.asarray(bond_score, dtype=float), "ForceBetweennessScore"
    else:
        raise ValueError("COLOR_BY must be 'force', 'betweenness', or 'score'")


/Users/tatiana/opt/anaconda3/envs/ovito-py/lib/python3.11/site-packages/ovito/_extensions/anari.py:2: UserWarning: Did you accidentally install the OVITO package from the PyPI repository in an Anaconda/Miniconda Python interpreter using the 'pip' command? This will likely lead to conflicts with existing libraries in the Anaconda environment, and import of the OVITO module may fail with an error related to the Qt framework. To fix this, please uninstall the ovito pip package by running 'pip uninstall -y ovito PySide6' and then install the OVITO Anaconda package provided by OVITO GmbH. Visit https://docs.ovito.org/python/introduction/installation.html for further instructions. If you would rather like to ignore this warning message, add the following code to the top of your Python script:

  import warnings
  warnings.filterwarnings('ignore', message='.*OVITO.*PyPI')

  import ovito._extensions.pyscript


In [ ]:
def modify(frame, data):
    contacts = read_lammps_local_contacts(CONTACT_FILE)

    B_df = compute_betweenness(contacts)
    contacts = contacts.merge(B_df, on=["i", "j"], how="left")
    contacts["B"] = contacts["B"].fillna(0.0)

    particle_ids = np.asarray(data.particles["Particle Identifier"], dtype=int)
    id_map = {pid: idx for idx, pid in enumerate(particle_ids)}

    # Force chains are selected by edge betweenness.
    threshold = np.percentile(contacts["B"], CHAIN_PERCENTILE)

    topology = []
    bond_force = []
    bond_betweenness = []
    bond_score = []

    for row in contacts.itertuples(index=False):
        if row.B >= threshold:
            i_id = int(row.i)
            j_id = int(row.j)

            if i_id in id_map and j_id in id_map:
                topology.append([id_map[i_id], id_map[j_id]])
                bond_force.append(float(row.F))
                bond_betweenness.append(float(row.B))
                bond_score.append(float(row.F * row.B))

    n_bonds = len(topology)

    # Pick which metric will be represented by Bond Type.
    metric_values, metric_name = choose_metric(
        COLOR_BY,
        bond_force,
        bond_betweenness,
        bond_score
    )

    bond_types, bin_edges = make_bond_types(metric_values, n_bins=N_BOND_TYPES)

    # Create actual OVITO bonds.
    bonds = data.particles_.create_bonds(count=n_bonds)

    bonds.create_property(
        "Topology",
        data=np.asarray(topology, dtype=np.int64)
    )

    # THIS is the important free-OVITO workaround:
    # Bond Type now encodes Force, Betweenness, or Force*BetweennessScore.
    bonds.create_property(
        "Bond Type",
        data=np.asarray(bond_types, dtype=np.int32)
    )

    # These properties are useful while in Python, but LAMMPS .data may not preserve them.
    # Bond Type is the property we rely on in OVITO Basic.
    bonds.create_property("Force", data=np.asarray(bond_force, dtype=float))
    bonds.create_property("Betweenness", data=np.asarray(bond_betweenness, dtype=float))
    bonds.create_property("ForceBetweennessScore", data=np.asarray(bond_score, dtype=float))
    
    
    
    # Make stronger / higher-type bonds thicker
    # This is not picked up either
    bond_radii_by_type = np.linspace(0.04, 0.25, N_BOND_TYPES)

    bond_radius = np.array([
        bond_radii_by_type[t - 1] for t in bond_types
    ], dtype=float)
    bonds.create_property("Radius", data=bond_radius)

    

    bonds.vis.enabled = True
    bonds.vis.width = 0.15
    data.particles.vis.radius = 0.0

    print("=" * 60)
    print(f"COLOR_BY: {COLOR_BY}  -> encoded in Bond Type")
    print(f"Metric: {metric_name}")
    print(f"Betweenness threshold: {threshold:.6g}")
    print(f"Force-chain bonds: {n_bonds}")
    print("Bond type bins:")
    for t in range(1, N_BOND_TYPES + 1):
        lo = bin_edges[t - 1]
        hi = bin_edges[t]
        print(f"  Type {t}: {lo:.6g} to {hi:.6g}")
    print("=" * 60)


In [ ]:
# =========================
# EXPORT THREE OVITO BASIC-FRIENDLY LAMMPS DATA FILES
# =========================
# Each file has identical force-chain bonds, but Bond Type encodes a different metric.

pipeline = import_file(PARTICLE_FILE)
pipeline.modifiers.append(modify)

outputs = {
    "force":       "force_chains_by_force.data",
    "betweenness": "force_chains_by_betweenness.data",
    "score":       "force_chains_by_score.data",
}

for mode, filename in outputs.items():
    COLOR_BY = mode
    print(f"\nExporting {filename} with Bond Type = binned {mode} ...")

    export_file(
        pipeline,
        filename,
        "lammps/data",
        atom_style="sphere"
    )

print("\nDone. Open one of the .data files in OVITO Basic.")
print("Then use: Add modification → Color coding → Operate on: Bonds → Input property: Bond Type")



Exporting force_chains_by_force.data with Bond Type = binned force ...
COLOR_BY: force  -> encoded in Bond Type
Metric: Force
Betweenness threshold: 0.0071824
Force-chain bonds: 498
Bond type bins:
  Type 1: 1.74019 to 3.27216
  Type 2: 3.27216 to 3.83409
  Type 3: 3.83409 to 4.31475
  Type 4: 4.31475 to 4.84128
  Type 5: 4.84128 to 5.3625
  Type 6: 5.3625 to 5.99111
  Type 7: 5.99111 to 6.68719
  Type 8: 6.68719 to 7.50113
  Type 9: 7.50113 to 9.15539
  Type 10: 9.15539 to 15.925

Exporting force_chains_by_betweenness.data with Bond Type = binned betweenness ...

Exporting force_chains_by_score.data with Bond Type = binned score ...

Done. Open one of the .data files in OVITO Basic.
Then use: Add modification → Color coding → Operate on: Bonds → Input property: Bond Type


## How to use in OVITO Basic

1. Open one of the exported files:
   - `force_chains_by_force.data`
   - `force_chains_by_betweenness.data`
   - `force_chains_by_score.data`

2. In OVITO:
   - `Add modification → Color coding`
   - `Operate on: Bonds`
   - `Input property: Bond Type`

3. Change the color gradient to `Viridis`, `Plasma`, or `Inferno` if you want a cleaner publication figure.

Recommendation: use `force_chains_by_score.data` for your main force-chain figure because the bond type encodes `Force × Betweenness`.
